# Medallion Architecture with PySpark & Iceberg
This notebook demonstrates a local data engineering pipeline using the TPC-H dataset.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Spark context is auto-configured by the tabulario/spark-iceberg image.
# We will ensure our session uses optimal configurations for small datasets (8GB RAM host).
spark = SparkSession.builder \
    .appName("MedallionArchitecture") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Show databases
spark.sql("SHOW DATABASES").show()


## 1. Setup Database
Create `prod_db` mapping to our Iceberg catalog.


In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS demo.prod_db")
spark.sql("USE demo.prod_db")



## 2. Bronze Layer: Raw Data Ingestion
Read CSV files and load them into Iceberg raw tables.


In [ ]:
# Load raw datasets
customer_df = spark.read.csv("/home/iceberg/data/customer.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("/home/iceberg/data/orders.csv", header=True, inferSchema=True)

# Write to Bronze layer
customer_df.writeTo("demo.prod_db.bronze_customer").tableProperty("format-version", "2").createOrReplace()
orders_df.writeTo("demo.prod_db.bronze_orders").tableProperty("format-version", "2").createOrReplace()

spark.sql("SHOW TABLES IN demo.prod_db").show()



## 3. Silver Layer: Cleaned Data
Cast columns, handle nulls, and simplify schema.


In [ ]:
# Read from bronze
customer_bronze = spark.table("demo.prod_db.bronze_customer")
orders_bronze = spark.table("demo.prod_db.bronze_orders")

# Transform customer
customer_silver = customer_bronze \
    .withColumnRenamed("c_custkey", "customer_id") \
    .withColumnRenamed("c_name", "customer_name") \
    .withColumnRenamed("c_mktsegment", "market_segment") \
    .select("customer_id", "customer_name", "market_segment")

# Transform orders
orders_silver = orders_bronze \
    .withColumnRenamed("o_orderkey", "order_id") \
    .withColumnRenamed("o_custkey", "customer_id") \
    .withColumnRenamed("o_totalprice", "total_price") \
    .withColumn("order_date", to_date(col("o_orderdate"))) \
    .select("order_id", "customer_id", "total_price", "order_date")

# Write to Silver layer
customer_silver.writeTo("demo.prod_db.silver_customer").tableProperty("format-version", "2").createOrReplace()
orders_silver.writeTo("demo.prod_db.silver_orders").tableProperty("format-version", "2").createOrReplace()



## 4. Gold Layer: Aggregated Data & Analytics
Create business-level views including Window functions and Joins.


In [ ]:
# We can also use Spark SQL directly
spark.sql("""
CREATE OR REPLACE TABLE demo.prod_db.gold_customer_revenue
USING iceberg
AS
SELECT 
    c.customer_id,
    c.customer_name,
    c.market_segment,
    SUM(o.total_price) as lifetime_value,
    COUNT(o.order_id) as total_orders
FROM demo.prod_db.silver_customer c
JOIN demo.prod_db.silver_orders o ON c.customer_id = o.customer_id
GROUP BY 1, 2, 3
""")

# Show aggregated data
spark.sql("SELECT * FROM demo.prod_db.gold_customer_revenue ORDER BY lifetime_value DESC LIMIT 5").show()



## 5. Advanced Transformations
Examples of Window functions and Broadcast Join optimization.


In [ ]:
# Top customers per market segment using Window
window_spec = Window.partitionBy("market_segment").orderBy(col("lifetime_value").desc())

ranked_customers = spark.table("demo.prod_db.gold_customer_revenue") \
    .withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") <= 3)

ranked_customers.show()



In [ ]:
# Broadcast Join Example
# Broadcasting the smaller customer table to optimize join
from pyspark.sql.functions import broadcast

customer_silver = spark.table("demo.prod_db.silver_customer")
orders_silver = spark.table("demo.prod_db.silver_orders")

# Provide join hint
optimized_join = orders_silver.join(broadcast(customer_silver), "customer_id", "inner")

optimized_join.explain()

